# Match Analysis — Pokemon TCG Agent Performance

Load aggregated Parquet tables from `make match-all` and explore agent behaviour,
win-rate trends, prize-race dynamics, decision patterns, and deck-type matchups.

## Setup


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

DATA = Path("data/matches/aggregated")


In [ ]:
try:
    results = pd.read_parquet(DATA / "results.parquet")
    frames = pd.read_parquet(DATA / "frames.parquet")
    turns = pd.read_parquet(DATA / "turn_summary.parquet")
    print(f"results: {len(results)} matches")
    print(f"frames:  {len(frames)} decision points")
    print(f"turns:   {len(turns)} turn-rows")
except FileNotFoundError:
    print("No aggregated data found. Run: make match-all")
    results = frames = turns = None


---
## 1. Win-Rate Matrix

Heatmap of agent A (row) vs agent B (column) win percentages.


In [ ]:
if results is not None:
    # Build win-rate matrix: agent0 vs agent1, % agent0 wins
    wins = results[results["winner"].isin([0, 1])].copy()
    wins["agent0_won"] = (wins["winner"] == 0).astype(int)

    matrix = wins.pivot_table(
        index="agent0", columns="agent1", values="agent0_won", aggfunc="mean"
    )

    plt.figure(figsize=(max(6, len(matrix.columns)), max(4, len(matrix.index))))
    sns.heatmap(matrix, annot=True, fmt=".1%", cmap="RdYlGn", vmin=0, vmax=1,
                linewidths=0.5, cbar_kws={"label": "Row agent win rate"})
    plt.title("Win Rate Matrix (Row vs Column)")
    plt.xlabel("Agent (as player 1)")
    plt.ylabel("Agent (as player 0)")
    plt.tight_layout()
    plt.show()


---
## 2. Action Frequency

What types of actions do agents take at each decision point?


In [ ]:
if frames is not None:
    OPT_NAMES = {
        0: "Num", 1: "Yes", 2: "No", 3: "Card", 7: "Play",
        8: "Attach", 9: "Evolve", 10: "Ability", 11: "Discard",
        12: "Retreat", 13: "Attack", 14: "End", 15: "Skill",
    }
    frames["action_name"] = frames["chosen_type"].map(OPT_NAMES).fillna("Other")

    # Merge agent names from results
    # frames don't carry agent names, so we infer from match_id prefix (approximate)
    m = frames[["match_id", "action_name"]].copy()
    # Just show overall distribution
    counts = frames["action_name"].value_counts()
    plt.figure(figsize=(10, 5))
    sns.barplot(x=counts.values, y=counts.index, hue=counts.index, palette="viridis", legend=False)
    plt.xlabel("Count")
    plt.title("Overall Action Distribution")
    plt.tight_layout()
    plt.show()


In [ ]:
if frames is not None and results is not None:
    # Merge agent names onto frames via match_id
    agent_map = pd.concat([
        results[["match_id", "agent0"]].rename(columns={"agent0": "agent"}),
        results[["match_id", "agent1"]].rename(columns={"agent1": "agent"}),
    ]).drop_duplicates("match_id")
    # Simple: first player in each frame is agent0, second is agent1
    f = frames.merge(results[["match_id", "agent0", "agent1"]], on="match_id", how="left")
    f["player_agent"] = np.where(f["player"] == 0, f["agent0"], f["agent1"])

    # Action distribution per agent (only MAIN decisions for clarity)
    main_actions = f[f["select_type"] == 0]
    action_by_agent = main_actions.groupby(["player_agent", "action_name"]).size().unstack(fill_value=0)
    action_pct = action_by_agent.div(action_by_agent.sum(axis=1), axis=0)

    plt.figure(figsize=(12, max(4, len(action_pct) * 0.4)))
    sns.heatmap(action_pct, annot=True, fmt=".0%", cmap="YlGnBu", linewidths=0.5)
    plt.title("Action Type Distribution per Agent (MAIN decisions only)")
    plt.tight_layout()
    plt.show()


---
## 3. Prize Race Dynamics

How does prize lead evolve over the course of a game?


In [ ]:
if turns is not None:
    turns["prize_lead"] = turns["me_prizes_taken"] - turns["opp_prizes_taken"]

    # Average prize lead over time (per player perspective)
    lead_by_turn = turns.groupby("turn")["prize_lead"].agg(["mean", "sem"])
    lead_by_turn = lead_by_turn[lead_by_turn.index <= 20]  # cap long games

    plt.figure(figsize=(10, 5))
    plt.plot(lead_by_turn.index, lead_by_turn["mean"], "b-", linewidth=2)
    plt.fill_between(
        lead_by_turn.index,
        lead_by_turn["mean"] - 1.96 * lead_by_turn["sem"],
        lead_by_turn["mean"] + 1.96 * lead_by_turn["sem"],
        alpha=0.2,
    )
    plt.axhline(0, color="gray", linestyle="--", alpha=0.5)
    plt.xlabel("Turn")
    plt.ylabel("Prize Lead (me - opp)")
    plt.title("Average Prize Lead Over Time")
    plt.tight_layout()
    plt.show()


In [ ]:
if turns is not None and results is not None:
    # First-prize advantage: does the player who takes first prize usually win?
    first_prize = turns[turns["prize_taken"]].copy()
    first_prize = first_prize.loc[first_prize.groupby("match_id")["turn"].idxmin()]
    fp = first_prize.merge(results[["match_id", "winner"]], on="match_id", how="left")
    fp["first_taker_won"] = (fp["player"] == fp["winner"]).astype(int)
    fp_wr = fp["first_taker_won"].mean()
    print(f"First-prize taker win rate: {fp_wr:.1%} ({fp['first_taker_won'].sum()}/{len(fp)})")

    # Comeback rate: % where trailing player after turn N goes on to win
    trail_counts = []
    for t in range(1, 13):
        trail = turns[turns["turn"] == t].copy()
        trail = trail.merge(results[["match_id", "winner"]], on="match_id", how="left")
        trail["trailing"] = trail["me_prizes_taken"] < trail["opp_prizes_taken"]
        trail["came_back"] = (trail["trailing"]) & (trail["player"] == trail["winner"])
        if len(trail) > 0:
            comeback_rate = trail["came_back"].sum() / trail["trailing"].sum()
            trail_counts.append({"turn": t, "comeback_rate": comeback_rate})

    if trail_counts:
        cb = pd.DataFrame(trail_counts)
        plt.figure(figsize=(8, 4))
        plt.plot(cb["turn"], cb["comeback_rate"], "ro-", linewidth=2)
        plt.xlabel("Turn")
        plt.ylabel("Comeback Rate")
        plt.title("Comeback Rate (trailing after turn N → win)")
        plt.tight_layout()
        plt.show()


---
## 4. Turn & Game Length Analysis


In [ ]:
if results is not None:
    clean = results[results["winner"].isin([0, 1])]
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    sns.histplot(clean["turns"], bins=range(0, clean["turns"].max() + 2, 2), kde=False)
    plt.xlabel("Total Turns")
    plt.title("Game Length Distribution")

    plt.subplot(1, 2, 2)
    by_agent = clean.groupby("agent0")["turns"].mean().sort_values()
    sns.barplot(x=by_agent.values, y=by_agent.index, hue=by_agent.index, palette="viridis", legend=False)
    plt.xlabel("Avg Turns (when agent0)")
    plt.title("Average Game Length by Agent")
    plt.tight_layout()
    plt.show()


---
## 5. First-Player Bias


In [ ]:
if results is not None and "first_player" in results.columns:
    fp_wr = results.groupby("first_player")["winner"].apply(
        lambda x: (x == x.name).mean() if len(x) > 0 else 0
    )
    print("Win rate by first player:")
    print(fp_wr.to_string(float_format="{:+.1%}".format))
else:
    print("first_player not available in aggregated data yet")


---
## 6. Phase Detection

Compare agent behaviour across early (prizes 6-4), mid (3-2), and late (1-0) game.


In [ ]:
if frames is not None:
    f = frames.copy()
    f["phase"] = pd.cut(
        f["me_prize_remaining"],
        bins=[-1, 0, 2, 4, 6],
        labels=["knockout", "late", "mid", "early"],
    )

    phase_actions = f[f["select_type"] == 0].groupby(["phase", "action_name"]).size().unstack(fill_value=0)
    phase_pct = phase_actions.div(phase_actions.sum(axis=1), axis=0)

    plt.figure(figsize=(10, 5))
    phase_pct.plot(kind="bar", stacked=True, colormap="tab10", ax=plt.gca())
    plt.title("Action Distribution by Game Phase")
    plt.xlabel("Phase")
    plt.ylabel("Proportion")
    plt.legend(title="Action", bbox_to_anchor=(1.05, 1))
    plt.tight_layout()
    plt.show()


---
## 7. Error Analysis


In [ ]:
if results is not None:
    errors = results[results["error"].notna()]
    print(f"Matches with errors: {len(errors)} / {len(results)}")
    if len(errors) > 0:
        print(errors[["agent0", "agent1", "error"]].to_string())


---
## 8. Decision Pattern Analysis

Dive into specific scenarios: when both Attack and Retreat are available, which is chosen?


In [ ]:
if frames is not None:
    # Decision: MAIN context, look at Attack vs Retreet vs End choices
    main = frames[frames["select_type"] == 0].copy()
    decision_counts = main.groupby(["match_id", "turn", "player"]).size().reset_index(name="decisions")
    print(f"Mean decisions per turn-player: {decision_counts['decisions'].mean():.2f}")
    print(f"Max decisions in a turn: {decision_counts['decisions'].max()}")

    # What % of decisions are "End" (pass)?
    end_pct = (main["chosen_type"] == 14).mean()
    print(f"\nEnd (pass) decision rate: {end_pct:.1%}")

    attack_pct = (main["chosen_type"] == 13).mean()
    print(f"Attack decision rate: {attack_pct:.1%}")


---
## 9. Deck-Type Matchup Analysis

Requires `card_db` to classify deck lists into archetypes.


In [ ]:
# Requires card_db and deck classification
# from pokemon.card_db import get_card_db
# db = get_card_db()
#
# def classify_deck(card_ids):
#     ...
#     return archetype
#
# if results is not None and "deck0" in results.columns:
#     results["arch0"] = results["deck0"].apply(classify_deck)
#     results["arch1"] = results["deck1"].apply(classify_deck)
#     arch_matrix = results.pivot_table(...)


---
## Summary & Next Steps

Key questions to explore:
- Which agents have the best prize-race curves?
- What actions differentiate winning vs losing games?
- Are there decision patterns that correlate with first-player advantage?
- Can we identify a "stall" pattern (too many End actions in mid-game)?
